# Modul Praktikum 1 — Sinyal, Sistem LTI, dan Konvolusi
**Mata Kuliah:** Sinyal dan Sistem (TKE1363) — Teknik Elektro, Universitas Jember

Modul ini mendampingi Minggu 1-3 (RPS revisi). Tujuan praktikum:
1. Membangkitkan dan memvisualisasikan sinyal dasar (unit step, impuls, sinusoid, eksponensial)
2. Memverifikasi sifat sistem (linearitas, time-invariance) secara numerik
3. Menghitung konvolusi (manual & `np.convolve`) dan menentukan respons sistem LTI

Setiap bagian ditutup dengan **Latihan Mahasiswa**.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (8, 3)

## Bagian A — Sinyal Dasar (Minggu 1)

In [2]:
n = np.arange(-10, 11)

def unit_step(n):
    return (n >= 0).astype(float)

def unit_impulse(n):
    return (n == 0).astype(float)

u = unit_step(n)
d = unit_impulse(n)

fig, axs = plt.subplots(1, 2, figsize=(10, 3))
axs[0].stem(n, u); axs[0].set_title('Unit Step u[n]'); axs[0].set_xlabel('n')
axs[1].stem(n, d); axs[1].set_title('Impuls Satuan δ[n]'); axs[1].set_xlabel('n')
plt.tight_layout(); plt.show()

In [3]:
t = np.linspace(0, 2, 500)
f0 = 3           # Hz
a = 1.5          # laju peluruhan

sinusoid = np.sin(2*np.pi*f0*t)
eksponensial = np.exp(-a*t)

fig, axs = plt.subplots(1, 2, figsize=(10, 3))
axs[0].plot(t, sinusoid); axs[0].set_title(f'Sinusoid {f0} Hz'); axs[0].set_xlabel('t (s)')
axs[1].plot(t, eksponensial); axs[1].set_title(f'Eksponensial e^(-{a}t)'); axs[1].set_xlabel('t (s)')
plt.tight_layout(); plt.show()

**Latihan A1.** Bangkitkan sinyal `x[n] = u[n] - u[n-5]` (pulsa persegi diskrit lebar 5) dan plot menggunakan `stem`.
Bandingkan energi sinyal ini dengan sinyal `x[n] = 0.8**n * u[n]` untuk n = 0..20 (hitung `np.sum(x**2)`).

## Bagian B — Verifikasi Sifat Sistem LTI (Minggu 2)

In [4]:
def sistem_contoh(x, n):
    """Sistem uji: y[n] = 2 x[n] - x[n-1]  (linear, time-invariant, kausal)"""
    y = np.zeros_like(x, dtype=float)
    for i in range(len(n)):
        y[i] = 2*x[i] - (x[i-1] if i-1 >= 0 else 0.0)
    return y

n = np.arange(0, 20)
x1 = np.sin(0.3*n)
x2 = np.cos(0.5*n)
a_, b_ = 2.0, -1.5

# Uji Linearitas: T{a x1 + b x2} =? a T{x1} + b T{x2}
lhs = sistem_contoh(a_*x1 + b_*x2, n)
rhs = a_*sistem_contoh(x1, n) + b_*sistem_contoh(x2, n)
print('Uji Linearitas — selisih maksimum:', np.max(np.abs(lhs - rhs)))
print('LINEAR' if np.allclose(lhs, rhs) else 'TIDAK LINEAR')

Uji Linearitas — selisih maksimum: 8.881784197001252e-16
LINEAR


In [5]:
# Uji Time-Invariance: y[n-n0] =? T{x[n-n0]}
n0 = 3
x_shift = np.zeros_like(x1)
x_shift[n0:] = x1[:-n0] if n0 > 0 else x1

y_asli = sistem_contoh(x1, n)
y_shift_dulu = sistem_contoh(x_shift, n)
y_shift_belakangan = np.zeros_like(y_asli)
y_shift_belakangan[n0:] = y_asli[:-n0]

selisih = np.max(np.abs(y_shift_dulu[n0:] - y_shift_belakangan[n0:]))
print('Uji Time-Invariance — selisih maksimum:', selisih)
print('TIME-INVARIANT' if selisih < 1e-9 else 'BUKAN TIME-INVARIANT')

Uji Time-Invariance — selisih maksimum: 0.0
TIME-INVARIANT


**Latihan B1.** Ganti `sistem_contoh` dengan sistem non-linear `y[n] = x[n]**2` lalu jalankan ulang uji linearitas —
apa yang terjadi pada selisihnya? Ulangi juga untuk sistem time-varying `y[n] = n * x[n]` dan uji time-invariance-nya.

## Bagian C — Konvolusi dan Respons Impuls (Minggu 3)

In [6]:
def konvolusi_manual(x, h):
    """Implementasi konvolusi diskrit y[n] = sum_k x[k] h[n-k] tanpa numpy.convolve"""
    Lx, Lh = len(x), len(h)
    y = np.zeros(Lx + Lh - 1)
    for i in range(Lx):
        for j in range(Lh):
            y[i+j] += x[i]*h[j]
    return y

x = np.array([1, 2, 3, 1])
h = np.array([1, 1, 1])   # respons impuls: moving-average orde-3 (tanpa normalisasi)

y_manual = konvolusi_manual(x, h)
y_numpy = np.convolve(x, h)

print('x       :', x)
print('h       :', h)
print('y manual:', y_manual)
print('y numpy :', y_numpy)
print('Cocok?  :', np.allclose(y_manual, y_numpy))

x       : [1 2 3 1]
h       : [1 1 1]
y manual: [1. 3. 6. 6. 4. 1.]
y numpy : [1 3 6 6 4 1]
Cocok?  : True


In [7]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3))
axs[0].stem(x); axs[0].set_title('x[n]')
axs[1].stem(h); axs[1].set_title('h[n] (respons impuls)')
axs[2].stem(y_numpy); axs[2].set_title('y[n] = x[n] * h[n]')
plt.tight_layout(); plt.show()

In [8]:
# Contoh sistem LTI nyata: filter moving-average sebagai penghalus sinyal berderau
np.random.seed(0)
n = np.arange(0, 60)
sinyal_bersih = np.sin(0.2*n)
derau = 0.3*np.random.randn(len(n))
sinyal_berderau = sinyal_bersih + derau

h_ma = np.ones(5)/5   # respons impuls moving-average orde-5
sinyal_halus = np.convolve(sinyal_berderau, h_ma, mode='same')

plt.plot(n, sinyal_berderau, label='Sinyal berderau (input)', alpha=0.6)
plt.plot(n, sinyal_halus, label='Output sistem LTI (moving average)', linewidth=2)
plt.plot(n, sinyal_bersih, '--', label='Sinyal asli (referensi)', alpha=0.7)
plt.legend(); plt.xlabel('n'); plt.title('Respons Sistem LTI melalui Konvolusi')
plt.tight_layout(); plt.show()

**Latihan C1 (untuk LKM1).** Pilih sebuah sinyal input x[n] (boleh sinyal sensor/audio sederhana buatan sendiri, panjang ≥ 30 sampel)
dan rancang respons impuls h[n] untuk salah satu tujuan berikut: (a) penghalus derau, (b) penonjol tepi/perubahan mendadak.
Hitung y[n] = x[n]*h[n] dengan `konvolusi_manual` **dan** `np.convolve`, buktikan hasilnya sama, lalu jelaskan interpretasi fisis hasil konvolusi tersebut.
Kumpulkan sebagai bagian dari LKM1.